In [1]:
from reachy_mini import ReachyMini
import numpy as np
import time

with ReachyMini() as mini:
    print("Connected!")


(<unknown>:78394): GStreamer-WARNING **: 20:12:48.766: External plugin loader failed. This most likely means that the plugin loader helper binary was not found or could not be run. You might need to set the GST_PLUGIN_SCANNER environment variable if your setup is unusual. This should normally not be required though.

(<unknown>:78394): GStreamer-WARNING **: 20:12:48.783: Failed to load plugin '/Users/aryanluthra/PersonalProjects/not-jarvis/.conda/lib/python3.11/site-packages/gstreamer_plugins/lib/gstreamer-1.0/libgstcurl.dylib': dlopen(/Users/aryanluthra/PersonalProjects/not-jarvis/.conda/lib/python3.11/site-packages/gstreamer_plugins/lib/gstreamer-1.0/libgstcurl.dylib, 0x0006): Symbol not found: _SSL_get0_group_name
  Referenced from: <D2B7064E-9013-3CA1-98B0-0A89988A2849> /Users/aryanluthra/PersonalProjects/not-jarvis/.conda/lib/python3.11/site-packages/gstreamer_plugins_libs/lib/libcurl.4.8.0.dylib
  Expected in:     <841E34D9-1ED9-3E2C-98F6-9DAB6B3C2963> /Users/aryanluthra/Persona

Connected!


In [ ]:
with ReachyMini() as mini:
    print("Connected to Reachy Mini! ")
    
    # Wiggle antennas
    print("Wiggling antennas...")
    mini.goto_target(antennas=[0.5, -0.5], duration=0.5)
    mini.goto_target(antennas=[-0.5, 0.5], duration=0.5)
    mini.goto_target(antennas=[0, 0], duration=0.5)

    print("Done!")

In [ ]:
with ReachyMini() as mini:
    print("Connected to Reachy Mini! ")

    print("Spinning Body...")
    mini.goto_target(body_yaw=0, duration=0.8)
    mini.goto_target(body_yaw=0.5, duration=0.8)
    mini.goto_target(body_yaw=-0.5, duration=0.8)
    mini.goto_target(body_yaw=0, duration=0.8)

    print("Done!")

In [ ]:
from scipy.spatial.transform import Rotation as R

with ReachyMini() as mini:
    print("Connected to Reachy Mini! ")

    print("Nodding Head...")

    # Build 4x4 pose matrices for head movements
    pose_down = np.eye(4)
    pose_down[:3, :3] = R.from_euler("xyz", [0, 15, 0], degrees=True).as_matrix()  # pitch down

    pose_up = np.eye(4)
    pose_up[:3, :3] = R.from_euler("xyz", [0, -15, 0], degrees=True).as_matrix()  # pitch up

    center = np.eye(4)  # identity = center position
    
    mini.goto_target(head=center, duration=0.5)
    for i in range(3):
        mini.goto_target(head=pose_down, duration=0.15)
        mini.goto_target(head=pose_up, duration=0.15)
    mini.goto_target(head=center, duration=0.5)

    print("Done!")

In [ ]:
with ReachyMini() as mini:
    print("Connected to Reachy Mini! ")

    print("Neck Dance...")

    pose_left = np.eye(4)
    pose_left[1, 3] = 0.03  # shift left

    pose_right = np.eye(4)
    pose_right[1, 3] = -0.03  # shift right

    center = np.eye(4)

    for i in range(3):
        mini.goto_target(head=pose_left, duration=0.2)
        mini.goto_target(head=pose_right, duration=0.2)
    mini.goto_target(head=center, duration=0.2)

    print("Done!")

In [ ]:
import tempfile
from openai import OpenAI
from time import sleep

tts_client = OpenAI()

def say(mini, text):
    """Speak text using OpenAI TTS through the Reachy's speaker."""
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    with tts_client.audio.speech.with_streaming_response.create(model="tts-1", voice="nova", input=text, response_format="wav") as response:
        response.stream_to_file(tmp.name)
    mini.media.play_sound(tmp.name)

with ReachyMini() as mini:
    say(mini, "Connected to Reachy Mini!")
    sleep(1)

    say(mini, "Looking straight ahead")
    mini.look_at_world(1, 0, 0, duration=1.5)

    say(mini, "Looking left")
    mini.look_at_world(1, 1, 0, duration=1.5)

    say(mini, "Looking right")
    mini.look_at_world(1, -1, 0, duration=1.5)

    say(mini, "Looking up")
    mini.look_at_world(1, 0, 0.5, duration=1.5)

    say(mini, "Looking down")
    mini.look_at_world(1, 0, -0.5, duration=1.5)

    say(mini, "Back to center")
    mini.look_at_world(1, 0, 0, duration=1.5)

    say(mini, "Looks like that worked! Done!")
    sleep(2)

In [ ]:
"""Idle animation — gaze, tilts, body follow, antennas, double-takes, room scans."""
import numpy as np
from scipy.spatial.transform import Rotation as R
import time
import random
import threading

def make_pose(yaw=0, pitch=0, roll=0):
    pose = np.eye(4)
    pose[:3, :3] = R.from_euler("xyz", [roll, pitch, yaw], degrees=True).as_matrix()
    return pose

def rand_signed(lo, hi):
    return random.uniform(lo, hi) * random.choice([-1, 1])

def idle(mini, duration=60, stop_event=None):
    if stop_event is None:
        stop_event = threading.Event()

    GAZE_YAW = (8, 15)
    GAZE_PITCH = (3, 8)
    GAZE_MOVE_DUR = 0.8
    GAZE_PAUSE = (4.0, 8.0)

    TILT = (5, 15)
    TILT_DUR = 0.4
    TILT_PAUSE = (2.0, 5.0)
    TILTS_MIN = 1
    TILTS_MAX = 3

    # Body follows gaze when yaw is large
    BODY_FOLLOW_THRESHOLD = 8    # degrees — body joins in past this
    BODY_FOLLOW_RATIO = 0.4      # body turns 40% of head yaw
    BODY_DUR = 0.6

    # Antennas
    ANTENNA_ALERT = [0.3, 0.3]   # perked up
    ANTENNA_RELAXED = [-0.1, -0.1]  # settled
    ANTENNA_DUR = 0.3

    # Double-take chance
    DOUBLE_TAKE_CHANCE = 0.15
    DOUBLE_TAKE_GLANCE_DUR = 0.3
    DOUBLE_TAKE_PAUSE = (0.3, 0.6)

    # Room scan chance
    SCAN_CHANCE = 0.10
    SCAN_DUR = 2.5               # slow sweep
    SCAN_YAW = (20, 30)          # wide sweep range
    SCAN_PAUSE = (1.0, 2.0)     # pause at end of sweep before locking on

    start = time.time()

    def alive():
        return not stop_event.is_set() and time.time() - start < duration

    while alive():
        roll = random.random()

        if roll < SCAN_CHANCE:
            # ── Room scan: slow sweep, then lock onto something ──
            sweep_yaw = rand_signed(*SCAN_YAW)
            mini.goto_target(
                head=make_pose(sweep_yaw, 0),
                body_yaw=sweep_yaw * BODY_FOLLOW_RATIO * 0.017,  # deg to rad approx
                antennas=ANTENNA_ALERT,
                duration=SCAN_DUR,
            )
            if not alive(): break
            time.sleep(random.uniform(*SCAN_PAUSE))

            # Lock onto a point within the sweep area
            yaw = sweep_yaw * random.uniform(0.3, 0.8)
            pitch = rand_signed(*GAZE_PITCH)
            mini.goto_target(
                head=make_pose(yaw, pitch),
                body_yaw=None,
                antennas=ANTENNA_RELAXED,
                duration=GAZE_MOVE_DUR,
            )
            if not alive(): break
            time.sleep(random.uniform(*GAZE_PAUSE))

        elif roll < SCAN_CHANCE + DOUBLE_TAKE_CHANCE:
            # ── Double-take: glance away, then snap back ──
            yaw = rand_signed(*GAZE_YAW)
            pitch = rand_signed(*GAZE_PITCH)

            # Quick glance in the opposite direction
            mini.goto_target(
                head=make_pose(-yaw * 0.5, pitch * 0.3),
                body_yaw=None,
                antennas=ANTENNA_RELAXED,
                duration=DOUBLE_TAKE_GLANCE_DUR,
            )
            if not alive(): break
            time.sleep(random.uniform(*DOUBLE_TAKE_PAUSE))

            # Snap back — "wait, what was that?"
            body_yaw = (yaw * BODY_FOLLOW_RATIO * 0.017) if abs(yaw) > BODY_FOLLOW_THRESHOLD else None
            mini.goto_target(
                head=make_pose(yaw, pitch),
                body_yaw=body_yaw,
                antennas=ANTENNA_ALERT,
                duration=DOUBLE_TAKE_GLANCE_DUR,
            )
            if not alive(): break
            time.sleep(random.uniform(*GAZE_PAUSE))

            # Settle antennas
            mini.goto_target(antennas=ANTENNA_RELAXED, body_yaw=None, duration=ANTENNA_DUR)

        else:
            # ── Normal gaze shift ──
            yaw = rand_signed(*GAZE_YAW)
            pitch = rand_signed(*GAZE_PITCH)

            # Antennas perk on shift
            body_yaw = (yaw * BODY_FOLLOW_RATIO * 0.017) if abs(yaw) > BODY_FOLLOW_THRESHOLD else None
            mini.goto_target(
                head=make_pose(yaw, pitch),
                body_yaw=body_yaw,
                antennas=ANTENNA_ALERT,
                duration=GAZE_MOVE_DUR,
            )
            if not alive(): break

            # Antennas relax after settling
            time.sleep(random.uniform(0.5, 1.5))
            mini.goto_target(antennas=ANTENNA_RELAXED, body_yaw=None, duration=ANTENNA_DUR)

            # Hold and stare
            time.sleep(random.uniform(*GAZE_PAUSE))

            # Head tilts while staring
            for _ in range(random.randint(TILTS_MIN, TILTS_MAX)):
                if not alive(): break
                tilt = rand_signed(*TILT)
                mini.goto_target(head=make_pose(yaw, pitch, tilt), body_yaw=None, duration=TILT_DUR)
                time.sleep(random.uniform(*TILT_PAUSE))

    # Return to center
    mini.goto_target(head=np.eye(4), body_yaw=0, antennas=[0, 0], duration=1.5)

# --- Run it ---
stop = threading.Event()

with ReachyMini() as mini:
    idle(mini, duration=7200, stop_event=stop)
    # To stop early from another cell: stop.set()

In [ ]:
# Run this cell to stop idle immediately
stop.set()

In [ ]:
"""Phase 1 test: Brain → Voice → AudioPlayer pipeline.
Type a message, hear it spoken. Type 'quit' to stop."""

import sys, os, logging
sys.path.insert(0, os.path.dirname(os.getcwd()))
from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

logging.basicConfig(level=logging.DEBUG, format="%(name)s | %(message)s")

from robot.orchestrator import Orchestrator

with Orchestrator() as orch:
    print("Pipeline running. Type a message:")
    while True:
        try:
            text = input("\nYou: ")
        except (EOFError, KeyboardInterrupt):
            break
        if text.strip().lower() in ("quit", "exit", "q"):
            break
        orch.send(text)
        # Wait for response to finish playing before prompting again
        import time
        time.sleep(1)
        while not orch.audio_q.empty():
            time.sleep(0.5)
        time.sleep(2)

print("Done.")